# W2 Lab — Prompting and Reasoning: Chain-of-Thought and Examples

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week02/W2_lab_prompting.ipynb)

**Goal.** Measure with code when three prompting techniques change the model's accuracy.

- **Chain-of-thought (CoT)** = making the model write its solution before the answer.
- **Few-shot prompting** = putting worked examples in the prompt for the model to imitate.
- **Self-consistency** = sampling several solutions and taking the majority answer.

Sections: setup (1) → answer-only vs worked solution (2) → reasoning that is not written (3) → a code-graded eval, baseline → format fix → your CoT prompt, target 11/12 (4) → worked examples (5) → self-consistency (6) → the limit of written reasoning (7) → in-class check (8).

*Runtime:* Google Colab, top to bottom, about 90 minutes. Every call is written out as in W1: a message list, one `create` call, the reply at `response.choices[0].message.content`.

This lab is not collected; the graded part of the week is the separate homework notebook `W2_hw_prompting.ipynb`. ✍️ marks a cell that needs your own writing. *Try:* names a prompt change to make and rerun before moving on.


## 1. Setup


### 1.1 Installation

`aisuite` runs the same code on OpenAI or Anthropic keys.

*Do:* run the cell (about 30 seconds, once per session).


In [ ]:
%pip install -q "aisuite[openai,anthropic]"

### 1.2 API key and model

**API key** = the secret string that identifies your account and bills usage to it (issuing steps: the API Setup guide on the course site). Do not share the notebook with the key inside.

*Do:* replace `PASTE-YOUR-KEY-HERE` with your key and run the cell. Nothing prints.


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"   # Anthropic accounts: MODEL = "anthropic:claude-haiku-4-5" and set ANTHROPIC_API_KEY instead

### 1.3 Client and a first call

The same call that opened W1. Every later cell repeats this shape; only the prompt string changes.

*Do:* run the cell. The output must be exactly `ready`.


In [ ]:
import aisuite

client = aisuite.Client()

messages = [{"role": "user", "content": "Reply with exactly: ready"}]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
response.choices[0].message.content

Any error here is a setup problem: recheck the API Setup guide before continuing.


## 2. Answer-Only versus Worked Solution

Same model, same problem, two prompts: one demands only the number, the other demands the solution first. If the scores differ, the difference comes from what the model wrote before the answer, not from the model.


### 2.1 The apple problem

The first example of the original chain-of-thought paper: a 2022 model answered 27 without a solution and 9 with one. This checks whether a current model still fails on it.

*Do:* run both cells. Both should be correct.


In [ ]:
## Answer only
APPLES = "A cafeteria has 23 apples. They use 20 for lunch and buy 6 more. How many apples do they have?"

messages = [
    {"role": "user", "content": APPLES + " Reply with only the number."},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)


In [ ]:
## Worked solution first
messages = [
    {"role": "user", "content": APPLES + " Write the solution step by step, then give the answer on the last line as ANSWER: <number>."},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)


Both are correct: one intermediate step (23 − 20 = 3) fits in a single prediction. The next cells add steps and digits to find where answer-only breaks.


### 2.2 Larger numbers, graded by code

Six problems of the same shape with a two-digit multiplication added. The question is a **template** = a string with named placeholders (`{a}`, `{b}`, ...) that `.format` fills from a table of numbers. Python computes each answer, so the score is exact. Expected: answer-only fails on most items, the worked solution passes.

*Do:* run the three cells and compare the two scores. Each line prints the correct answer, then the model's reply.


In [ ]:
QUESTION = "A warehouse holds {a} crates with {b} items each. {c} items are shipped out and {d} items are returned. How many items are in the warehouse now?"

numbers = [
    (47, 38, 519, 284),
    (63, 29, 807, 356),
    (84, 57, 2046, 173),
    (92, 73, 3318, 907),
    (54, 96, 1480, 622),
    (79, 68, 4155, 231),
]

problems = []
for a, b, c, d in numbers:
    question = QUESTION.format(a=a, b=b, c=c, d=d)
    answer = a * b - c + d
    problems.append((question, answer))

for question, answer in problems:
    print(answer, question)


In [ ]:
## Answer only
INSTRUCTION = "Reply with only the number."

score_answer_only = 0
for question, answer in problems:
    messages = [
        {"role": "user", "content": question + " " + INSTRUCTION},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    output = output.replace(",", "").strip()
    print(answer, output)
    if output == str(answer):
        score_answer_only += 1

print("answer only:", score_answer_only, "/", len(problems))


In [ ]:
## Worked solution first
INSTRUCTION = "Write the solution step by step, then give the answer on the last line as ANSWER: <number>."

score_worked = 0
for question, answer in problems:
    messages = [
        {"role": "user", "content": question + " " + INSTRUCTION},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    last_line = output.strip().splitlines()[-1].replace(",", "")
    print(answer, last_line)
    if str(answer) in last_line:
        score_worked += 1

print("worked solution:", score_worked, "/", len(problems))


The gap is the chain-of-thought effect: intermediate values written into the text before the answer.

*Try:* make `crates` and `items` one-digit and rerun; the gap should close. Then make them three-digit; the worked solution begins to fail too, because each written step also has a size limit.


## 3. Reasoning That Is Not Written

Two controls on the same six problems. Each keeps the instruction to reason but removes the reasoning from the text before the answer. If both score like answer-only, the accuracy in Section 2 came from the written text, not from the instruction to think.


### 3.1 Thinking in silence

The prompt asks the model to think carefully and write only the number.

*Do:* run the cell. Expected score: the answer-only level.


In [ ]:
## Told to think silently
INSTRUCTION = "Think about it carefully in silence, do not write your reasoning, and reply with only the number."

score_silent = 0
for question, answer in problems:
    messages = [
        {"role": "user", "content": question + " " + INSTRUCTION},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    output = output.replace(",", "").strip()
    print(answer, output)
    if output == str(answer):
        score_silent += 1

print("silent thinking:", score_silent, "/", len(problems))
print("answer only:", score_answer_only, "  worked solution:", score_worked)


Reasoning that is not written is not performed.


### 3.2 Answer first, solution after

The prompt asks for the full solution, but after the answer line. Everything is written; nothing is written before the answer.

*Do:* run the cell. Expected score: the answer-only level.


In [ ]:
## Answer first, then the solution
INSTRUCTION = "Give the answer on the first line as ANSWER: <number>, then explain the solution step by step."

score_answer_first = 0
for question, answer in problems:
    messages = [
        {"role": "user", "content": question + " " + INSTRUCTION},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    first_line = output.strip().splitlines()[0].replace(",", "")
    print(answer, first_line)
    if str(answer) in first_line:
        score_answer_first += 1

print("answer first:", score_answer_first, "/", len(problems))
print("answer only:", score_answer_only, "  worked solution:", score_worked)


The answer is produced before any intermediate value exists, and a token already emitted cannot be revised. The original chain-of-thought paper reports the same. Rule for every prompt from here on: the solution goes before the answer, and the grader reads the answer from the end of the reply (an `ANSWER:` last line, or `<answer>` tags after `<thinking>`).

*Try:* add `"If the solution reaches a different number, end with CORRECTION: <number>."` to the answer-first prompt and count how often a correction appears.


## 4. A Code-Graded Evaluation

**Code-graded evaluation** = running a prompt over a fixed test set with known answers and grading the outputs with code. The task: how many legs the animal in a statement has. Some statements are tricky (a fox that lost a leg and regrew two). The section improves one prompt in three versions and scores each.


### 4.1 The eval set

**Eval set** = the fixed list of test items the prompt is scored on. Here: twelve statements, each with a **golden answer** = the known correct output.

*Do:* run the cell and mark the statements you expect the model to miss.


In [ ]:
eval_data = [
    {"animal_statement": "The animal is a human.", "golden_answer": "2"},
    {"animal_statement": "The animal is a snake.", "golden_answer": "0"},
    {"animal_statement": "The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.", "golden_answer": "5"},
    {"animal_statement": "The animal is a dog.", "golden_answer": "4"},
    {"animal_statement": "The animal is a cat with two extra legs.", "golden_answer": "6"},
    {"animal_statement": "The animal is an elephant.", "golden_answer": "4"},
    {"animal_statement": "The animal is a bird.", "golden_answer": "2"},
    {"animal_statement": "The animal is a fish.", "golden_answer": "0"},
    {"animal_statement": "The animal is a spider with two extra legs", "golden_answer": "10"},
    {"animal_statement": "The animal is an octopus.", "golden_answer": "8"},
    {"animal_statement": "The animal is an octopus that lost two legs and then regrew three legs.", "golden_answer": "9"},
    {"animal_statement": "The animal is a two-headed, eight-legged mythical creature.", "golden_answer": "8"},
]
print(len(eval_data), "items")

### 4.2 The first prompt, on one item

The prompt is a template; `.format` fills the statement into the `<animal_statement>` tags. **Delimiters** = markers that separate pasted data from instructions. This cell sends one item (the fox) to see the raw reply before scoring the whole set.

*Do:* run the cell. Note the reply's shape (bare number or sentence) and value (5 is correct).


In [ ]:
PROMPT_V1 = """You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{statement}</animal_statement>

How many legs does the animal have? Please respond with a number"""

item = eval_data[2]   # the fox

messages = [
    {"role": "user", "content": PROMPT_V1.format(statement=item["animal_statement"])},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
output = response.choices[0].message.content
output

*Try:* set `item` to `eval_data[0]` and `eval_data[10]`. Does the reply's shape stay the same?


### 4.3 The whole set, graded

The loop repeats the call of 4.2 for every item; the second cell grades the replies.

*Do:* run both cells.


In [ ]:
outputs_v1 = []
for item in eval_data:
    messages = [
        {"role": "user", "content": PROMPT_V1.format(statement=item["animal_statement"])},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    outputs_v1.append(response.choices[0].message.content)

for item, output in zip(eval_data, outputs_v1):
    print("golden", item["golden_answer"], "| output:", output)

The grader is one exact comparison. A correct number inside a sentence grades as wrong.


In [ ]:
score_v1 = 0
for item, output in zip(eval_data, outputs_v1):
    correct = output.strip() == item["golden_answer"]
    score_v1 += correct
    print(correct, "golden", item["golden_answer"], "| output:", output)

print("score:", score_v1, "/ 12")


Two kinds of miss: sentences instead of bare numbers (format), and wrong numbers on tricky items (reasoning). The next two prompts fix them one at a time.


### 4.4 Fixing the format

One changed last line: "Respond only with a numeric digit, like 2 or 6, and nothing else."

*Do:* run the cell. Format misses should disappear; tricky items should still miss.


In [ ]:
PROMPT_V2 = """You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{statement}</animal_statement>

How many legs does the animal have? Respond only with a numeric digit, like 2 or 6, and nothing else."""

outputs_v2 = []
for item in eval_data:
    messages = [
        {"role": "user", "content": PROMPT_V2.format(statement=item["animal_statement"])},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    outputs_v2.append(response.choices[0].message.content)

score_v2 = 0
for item, output in zip(eval_data, outputs_v2):
    correct = output.strip() == item["golden_answer"]
    score_v2 += correct
    print(correct, "golden", item["golden_answer"], "| output:", output)

print("score:", score_v2, "/ 12")

*Try:* soften the last line to `"Respond with the number."` and rerun. Which items come back as sentences?


### 4.5 Chain of thought, graded ✍️

The remaining misses are reasoning misses; Section 2's fix applies. The prompt must make the model reason inside `<thinking>` tags first and put only the integer inside `<answer>` tags. The grader then extracts the integer from the tags.

**Starter** = the prefilled value of a fill-in; it runs but does not pass.

*Do:* write `PROMPT_V3`: keep the task statement and `{statement}`; replace the closing question with the two-part instruction. Then run the three cells below.

Hints: name both tags; say what goes in each; state that `<answer>` holds the number and nothing else (the extraction takes the tag content verbatim, so "5 legs" fails).


In [ ]:
### FILL IN (START) ###
# Keep the first four lines of the prompt (task statement, <animal_statement> tags).
# Replace the last line with two instructions:
#   1. reason step by step inside <thinking> tags
#   2. put the final answer, the integer alone, inside <answer> tags
PROMPT_V3 = """You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{statement}</animal_statement>

How many legs does the animal have?"""
### FILL IN (END) ###

One item first. The reply should be a `<thinking>` block followed by an `<answer>` block.


In [ ]:
item = eval_data[2]   # the fox

messages = [
    {"role": "user", "content": PROMPT_V3.format(statement=item["animal_statement"])},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
output = response.choices[0].message.content
print(output)

The reply can no longer be compared whole with `"5"`. **Regular expression** = a text pattern that finds strings of a given shape; here it takes what sits between the answer tags.


In [ ]:
import re

match = re.search(r"<answer>(.*?)</answer>", output, re.DOTALL)
extracted = None
if match:
    extracted = match.group(1).strip()

print(match)
print("extracted:", extracted)


The extraction is now inside the loop. Target: **at least 11 of 12**.

*Do:* run the cell; edit `PROMPT_V3` and rerun the three cells until the target is reached.


In [ ]:
outputs_v3 = []
for item in eval_data:
    messages = [
        {"role": "user", "content": PROMPT_V3.format(statement=item["animal_statement"])},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    outputs_v3.append(response.choices[0].message.content)

cot_score = 0
for item, output in zip(eval_data, outputs_v3):
    match = re.search(r"<answer>(.*?)</answer>", output, re.DOTALL)
    extracted = None
    if match:
        extracted = match.group(1).strip()
    correct = extracted == item["golden_answer"]
    cot_score += correct
    print(correct, "golden", item["golden_answer"], "extracted", extracted, "|", item["animal_statement"])

print("score:", cot_score, "/ 12   target 11")

The sequence baseline → format fix → reasoning fix, each scored, is how prompts are improved in practice.

*Try:* remove the words restricting `<answer>` to the number alone; the extraction picks up "5 legs". Then put the answer tags before the thinking tags; the tricky items return to their 4.4 values (Section 3.2).


### 4.6 The price of thinking

Each response carries a `usage` field with the billed token counts. **Completion tokens** = the tokens the model generated in its reply. The two cells send the fox item under the direct prompt and under the CoT prompt.

*Do:* run both cells and compare completion tokens.


In [ ]:
## Direct answer
messages = [
    {"role": "user", "content": PROMPT_V2.format(statement=eval_data[2]["animal_statement"])},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)
response.usage.completion_tokens

In [ ]:
## Chain of thought
messages = [
    {"role": "user", "content": PROMPT_V3.format(statement=eval_data[2]["animal_statement"])},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)
response.usage.completion_tokens

The accuracy of 4.5 was bought with output tokens. **Test-time compute** = accuracy bought with more generation at answer time, the weights unchanged. Section 6 spends more of it.


## 5. Learning from Examples

**Few-shot prompting** = example question–answer pairs (**exemplars**) in the prompt; the model imitates their format, tone, and procedure. **Zero-shot** = no examples in the prompt.


### 5.1 The parent bot

Asked without an example, the model answers a child's question like an encyclopedia. One worked example should change the tone without any instruction describing it.

*Do:* run both cells and compare the tone.


In [ ]:
## Asked cold
messages = [
    {"role": "user", "content": "Will Santa bring me presents on Christmas?"},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

In [ ]:
## One worked example in the prompt
messages = [
    {"role": "user", "content": """Please complete the conversation by writing the next line, speaking as "A".
Q: Is the tooth fairy real?
A: Of course, sweetie. Wrap up your tooth and put it under your pillow tonight. There might be something waiting for you in the morning.
Q: Will Santa bring me presents on Christmas?"""},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

One example fixed tone and length; no instruction described either.

*Try:* rewrite the example answer as a curt parent (`"A: No. The tooth fairy is a story."`) and rerun. Then add a second warm Q/A pair and check whether two examples hold the tone more firmly.


### 5.2 Email classification by examples ✍️

Categories: (A) Pre-sale question, (B) Broken or defective item, (C) Billing question, (D) Other. The grader checks the **last character** of the output against the correct letter. The starter prompt does not name the categories, so it scores 0 of 4. Target: **4 of 4**.

*Do:* rewrite `EMAIL_PROMPT` with the category list and a few example emails answered in the same closing line; run and iterate until all four read `PASS`.

Hints: two or three example emails (not the ones in `EMAILS`), each answered `The correct category is: B`; end the prompt with `The correct category is:` left open.


In [ ]:
### FILL IN (START) ###
# Replace the whole prompt. It must contain:
#   1. the four categories (A)-(D)
#   2. two or three example emails, each answered with the line: The correct category is: <letter>
#   3. the placeholder {email}, then the same line left open: The correct category is:
EMAIL_PROMPT = """Please classify this email as either green or blue: {email}"""
### FILL IN (END) ###

EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.",  # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?",  # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???",  # (C) Billing question
    "How did I get here I am not good with computer.  Halp.",  # (D) Other (please explain)
]
ANSWERS = [["B"], ["A", "D"], ["C"], ["D"]]

email_score = 0
for email, accepted in zip(EMAILS, ANSWERS):
    messages = [
        {"role": "user", "content": EMAIL_PROMPT.format(email=email)},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    correct = output.strip()[-1] in accepted
    email_score += correct
    print(correct, "expected", accepted, "| output:", output)

print("score:", email_score, "/ 4")

*Try:* delete the examples and keep only the category list and the closing line. Does the format survive? Then add a fifth email of your own to `EMAILS` and `ANSWERS`.

Exemplars whose answers are step-by-step solutions are few-shot CoT; the homework asks you to write them.


## 6. Self-Consistency

**Self-consistency** = sampling several reasoning paths at nonzero temperature and taking the majority of the extracted answers. Wrong paths scatter; correct paths agree.

**Temperature** = the sampling-randomness parameter of generation; 0.0 always takes the most probable token, 1.0 samples from the model's distribution.


### 6.1 One sample at temperature 1.0

The call of 4.5 on the fox with `temperature=1.0`.

*Do:* run the cell three or four times and note the extracted answers.


In [ ]:
HARD = eval_data[2]   # the fox that lost a leg and regrew two

messages = [
    {"role": "user", "content": PROMPT_V3.format(statement=HARD["animal_statement"])},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=1.0)
output = response.choices[0].message.content
match = re.search(r"<answer>(.*?)</answer>", output, re.DOTALL)
extracted = None
if match:
    extracted = match.group(1).strip()

print(output)
print("extracted:", extracted)

At temperature 1.0 the path differs from run to run, and on a tricky item the answer sometimes differs too.


### 6.2 Five samples

The loop repeats 6.1 five times and keeps the extracted answers.

*Do:* run the cell.


In [ ]:
N_SAMPLES = 5

samples = []
for i in range(N_SAMPLES):
    messages = [
        {"role": "user", "content": PROMPT_V3.format(statement=HARD["animal_statement"])},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=1.0)
    output = response.choices[0].message.content
    match = re.search(r"<answer>(.*?)</answer>", output, re.DOTALL)
    if match:
        samples.append(match.group(1).strip())

samples

### 6.3 The vote

`Counter` tallies the samples; the most common value is the answer.

*Do:* run the cell and compare the majority with the golden answer.


In [ ]:
from collections import Counter

votes = Counter(samples)
majority = None
if votes:
    majority = votes.most_common(1)[0][0]

print("votes:", dict(votes))
print("majority:", majority, "  golden:", HARD["golden_answer"])


One sample stands or falls with one path; the vote averages over paths at the price of five calls (test-time compute).

*Try:* set `temperature=0.0` in 6.2 and rerun 6.2 and 6.3; the five samples collapse to one path. Restore 1.0 and set `N_SAMPLES = 9`. Then point `HARD` at `eval_data[0]`: on an item every path gets right, the vote is five calls for nothing.


## 7. The Limit of Written Reasoning

**Hallucination** = generating content that is plausible but not factual. Chain-of-thought repairs computation, not knowledge. When the model lacks a fact, writing more does not produce it; it invents a plausible premise and reasons on top of it. The question below has no reliable answer in any source.

*Do:* run the three cells and compare what each prompt produces.


In [ ]:
## Asked cold
messages = [
    {"role": "user", "content": "Who is the heaviest hippo of all time?"},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

In [ ]:
## With chain of thought
messages = [
    {"role": "user", "content": "Who is the heaviest hippo of all time? "
                                "Think step by step in <thinking> tags first, then give your answer."},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

In [ ]:
## With an out
messages = [
    {"role": "user", "content": "Who is the heaviest hippo of all time? "
                                "Only answer if you know the answer with certainty."},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

The chain-of-thought version reasons in good form toward a name it cannot know. **Giving an out** = an instruction that makes declining an acceptable answer. Checking a premise against the world needs a tool (W3).

*Try:* replace the out with `"If you are not sure, answer exactly: I don't know."` Then ask, with the same out, a question the model does know ("What is the largest land animal alive today?") and confirm the out does not suppress a known answer.


## 8. In-class check

The two fill-ins of this lab, 4.5 and 5.2, are practice; nothing from this notebook is submitted. The cell prints their state.

*Do:* run the cell.


In [ ]:
checks = {
    "4.5 chain-of-thought prompt reaches >= 11/12": cot_score >= 11,
    "5.2 email prompt reaches 4/4":                  email_score == 4,
}

for item, ok in checks.items():
    if ok:
        print("PASS ", item)
    else:
        print("FAIL ", item)


---

Homework: `W2_hw_prompting.ipynb`, collected, due before the W3 class.

W3: plain Python functions become tools the model can call.
